In [1]:
import pandas as pd
import numpy as np

df =pd.read_csv("F:\\PYTHON PROJ\\LEARN PYTHON\\Customer Churn Prediction\\data after ESA.csv")

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgCharges,tenure_group,high_charge,num_of_service
0,0,0,1,0,1,0,0,DSL,0,1,...,Month-to-month,1,Electronic check,29.85,29.85,0,29.850000,0-6m,0,2
1,1,0,0,0,34,1,0,DSL,1,0,...,One year,0,Mailed check,56.95,1889.50,0,55.573529,2-3y,0,4
2,1,0,0,0,2,1,0,DSL,1,1,...,Month-to-month,1,Mailed check,53.85,108.15,1,54.075000,0-6m,0,4
3,1,0,0,0,45,0,0,DSL,1,0,...,One year,0,Bank transfer (automatic),42.30,1840.75,0,40.905556,3-5y,0,4
4,0,0,0,0,2,1,0,Fiber optic,0,0,...,Month-to-month,1,Electronic check,70.70,151.65,1,75.825000,0-6m,1,2


Step 1. Data reengineering to improve random forest performance 

We need to add feature charge_per_tenure = TotalCharges/tenure

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   int64  
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   int64  
 3   Dependents        7043 non-null   int64  
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   int64  
 6   MultipleLines     7043 non-null   int64  
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   int64  
 9   OnlineBackup      7043 non-null   int64  
 10  DeviceProtection  7043 non-null   int64  
 11  TechSupport       7043 non-null   int64  
 12  StreamingTV       7043 non-null   int64  
 13  StreamingMovies   7043 non-null   int64  
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   int64  
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [3]:
df['charge_per_tenure']= df['TotalCharges']/df['tenure']

In [4]:
print(np.isinf(df['charge_per_tenure']).sum())


11


there is inf value due to division  

In [5]:
df['charge_per_tenure'] = df.charge_per_tenure.replace([np.inf], np.nan)

df['charge_per_tenure'] = df.charge_per_tenure.fillna(df.charge_per_tenure.mean())

In [6]:
print(np.isinf(df['charge_per_tenure']).sum())

0


The second feature we need to add is an indicator telling if the customer is high value

In [7]:
df['is_high_value']= (df['TotalCharges'] >= df.TotalCharges.median()).astype(int)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             7043 non-null   int64  
 1   SeniorCitizen      7043 non-null   int64  
 2   Partner            7043 non-null   int64  
 3   Dependents         7043 non-null   int64  
 4   tenure             7043 non-null   int64  
 5   PhoneService       7043 non-null   int64  
 6   MultipleLines      7043 non-null   int64  
 7   InternetService    7043 non-null   str    
 8   OnlineSecurity     7043 non-null   int64  
 9   OnlineBackup       7043 non-null   int64  
 10  DeviceProtection   7043 non-null   int64  
 11  TechSupport        7043 non-null   int64  
 12  StreamingTV        7043 non-null   int64  
 13  StreamingMovies    7043 non-null   int64  
 14  Contract           7043 non-null   str    
 15  PaperlessBilling   7043 non-null   int64  
 16  PaymentMethod      7043 non-null   

Step 2 Redo the one hot encoding

In [9]:
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns

cat_cols

Index(['InternetService', 'Contract', 'PaymentMethod', 'tenure_group'], dtype='str')

In [10]:
df=pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [11]:
bool_cols = df.select_dtypes(include=['boolean']).columns
df[bool_cols] = df[bool_cols].astype(int)   

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 34 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   MultipleLines                          7043 non-null   int64  
 7   OnlineSecurity                         7043 non-null   int64  
 8   OnlineBackup                           7043 non-null   int64  
 9   DeviceProtection                       7043 non-null   int64  
 10  TechSupport                            7043 non-null   int64  
 11  StreamingTV    

In [13]:
df.to_csv('churn_cleaned and reengi data.csv', index=False)

In [14]:
df.dtypes
df.describe()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 34 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   MultipleLines                          7043 non-null   int64  
 7   OnlineSecurity                         7043 non-null   int64  
 8   OnlineBackup                           7043 non-null   int64  
 9   DeviceProtection                       7043 non-null   int64  
 10  TechSupport                            7043 non-null   int64  
 11  StreamingTV    

In [15]:
X = df.drop('Churn',axis=1)
y=df['Churn']

Step 2. Train/Test Split

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state= 42, stratify=y)


In [17]:
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(5634, 33)
(1409, 33)
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


2.2 train use fit and transform operation. 
test use transform operation

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [19]:
print(np.mean(X_train[:,0]))
print(np.std(X_train[:,0]))

1.1350522935464859e-16
1.0


mean ≈ 0
std ≈ 1

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, f1_score 

model = LogisticRegression(max_iter=4000)
model.fit(X_train, y_train)

y_pred= model.predict(X_test)
y_prob =model.predict_proba(X_test)[:,1]

LOGISTICS REG SEEMS EVEN WORESE THAN BEFORE

In [21]:
cm= confusion_matrix(y_test, y_pred)

cm

array([[932, 103],
       [176, 198]])

In [22]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.71      0.73      1409
weighted avg       0.79      0.80      0.79      1409



In [23]:
thresholds = np.arange(0.1, 0.9, 0.005)
best_threashhold= 0
best_f1=0

for t in thresholds:
    y_pred_t = (y_prob>t).astype(int)
    f_1 = f1_score(y_test, y_pred_t)

    if(f_1 > best_f1):
        best_f1=f_1
        best_threashhold=t
    
print(f"Best threshold:{best_threashhold:.2f}")
print(f"Best f1 score: {best_f1:.2f}")    


Best threshold:0.28
Best f1 score: 0.63


In [24]:
from sklearn.metrics  import  recall_score
best_rthreashhold= 0
best_recall=0

for t in thresholds:
    y_pred_t = (y_prob>t).astype(int)
    recall = recall_score(y_test, y_pred_t)

    if(recall > best_recall):
        best_recall=recall
        best_rthreashhold=t
    
print(f"Best r_threshold:{best_rthreashhold:.2f}")
print(f"Best recall score: {best_recall:.2f}")    

Best r_threshold:0.10
Best recall score: 0.95


Now use RF 

In [25]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X,y, test_size = 0.2, random_state= 42, stratify=y)

In [28]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    max_features='sqrt',
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train_r, y_train_r)

y_pred_rf = rf.predict(X_test_r)
y_prob_rf = rf.predict_proba(X_test_r)[:,1]

In [29]:
print(confusion_matrix(y_test_r, y_pred_rf))
print("Random Forest")
print(classification_report(y_test_r, y_pred_rf))

[[825 210]
 [111 263]]
Random Forest
              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1035
           1       0.56      0.70      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.80      0.77      0.78      1409



we can find now the positive recall is better than the one without reeng of feature

In [30]:
from sklearn.metrics  import  recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

best_RF_threashhold= 0
best_RF_recall=0

best_RF_f1 =0
best_RF_f1threashhold= 0


for t in thresholds:
    y_pred_t = (y_prob_rf>t).astype(int)
    recall = recall_score(y_test_r, y_pred_t)
    f1 = f1_score(y_test_r, y_pred_t)
    precision = precision_score(y_test_r, y_pred_t)
    accuracy = accuracy_score(y_test_r, y_pred_t)

    print(f"t={t:.2f}| recall = {recall:.3f}| f1={f1:.3f}| precsion={precision:.3f}| accuracy={accuracy:.3f}" )
          
    if(recall > best_RF_recall):
        best_RF_recall=recall
        best_RF_threashhold=t
    
    if(f1>best_RF_f1):
        best_RF_f1threashhold=t
        best_RF_f1 = f1


print(f"\n Best r_f1_threshold:{best_RF_f1threashhold:.3f}")
print(f"Best f1 score: {best_RF_f1:.3f}")    

print(f"\n Best r_threshold:{best_RF_threashhold:.3f}")
print(f"Best recall score: {best_RF_recall:.3f}")  

t=0.10| recall = 0.973| f1=0.515| precsion=0.350| accuracy=0.513
t=0.11| recall = 0.973| f1=0.517| precsion=0.352| accuracy=0.517
t=0.11| recall = 0.971| f1=0.522| precsion=0.357| accuracy=0.527
t=0.12| recall = 0.968| f1=0.522| precsion=0.358| accuracy=0.530
t=0.12| recall = 0.965| f1=0.526| precsion=0.361| accuracy=0.538
t=0.13| recall = 0.963| f1=0.529| precsion=0.364| accuracy=0.544
t=0.13| recall = 0.955| f1=0.530| precsion=0.367| accuracy=0.551
t=0.14| recall = 0.955| f1=0.533| precsion=0.370| accuracy=0.556
t=0.14| recall = 0.952| f1=0.536| precsion=0.373| accuracy=0.563
t=0.15| recall = 0.949| f1=0.541| precsion=0.378| accuracy=0.572
t=0.15| recall = 0.949| f1=0.546| precsion=0.383| accuracy=0.581
t=0.16| recall = 0.949| f1=0.549| precsion=0.386| accuracy=0.586
t=0.16| recall = 0.949| f1=0.553| precsion=0.391| accuracy=0.593
t=0.17| recall = 0.947| f1=0.556| precsion=0.393| accuracy=0.598
t=0.17| recall = 0.944| f1=0.559| precsion=0.397| accuracy=0.605
t=0.18| recall = 0.939| f

Very slight improving for random forest with feature re-eng. We need to have another model to improve.